# board

> The screen-guide board: a closed drawing language over a page, the check of each line, the page context and the stream of steps.

A learner looks at a page and says "I don't get alpha bar". A tutor answers on the page, as a teacher does at a whiteboard. It circles the
term, draws an arrow to a label in the figure, moves the slider of the demo, fills a small worked example in a side panel, and
speaks one short line for each step.

This module is the part of the board that knows nothing about learning:

- the **board language**: a closed list of drawing commands, each tied to a thing on the page (an *anchor*), and the check of
  each line of the model's reply;
- the **context**: what the page shows, cleaned and cut to size;
- the **video cues**: a quote from the transcript of a video, at its time;
- the **stream**: each step goes to the page as soon as it is whole.

`static/board.js` draws a checked board on the page, and `static/texparts.js` gives a formula the ids of its parts. abhyasa and Leela
both use this module and these files. The decisions about learning stay in abhyasa: the tutor tiers (tutor, then vault, then
web), the trust rules of sources, and the prompts.

In [ ]:
#| default_exp board

In [ ]:
#| hide
from fastcore.test import *

Import the helpers. The module needs only fastcore.

In [ ]:
#| export
import json, math, re
from itertools import takewhile
from fastcore.utils import *
from rupaka.spec import PKG

## The files

A host serves the files of the board. `assets` gives their paths. `bundle('js')` is `texparts.js` and `board.js` as one script, and
`bundle('css')` is the paper tokens and `board.css` as one style sheet. The pen libraries (Rough.js and perfect-freehand, MIT) are
in the folder `libs`. The board loads them on its first use from the URL folder `host.assets` (default `/vendor/`), so a host
serves `libs` there. The board's side panel uses the hand font Patrick Hand: a host loads it with `rupaka.render.font_css`.

The script sets `window.rupakaBoard` and `window.rupakaTex`. It sets the old names `window.abhyasaBoard` and `window.abhyasaTex`
to the same objects, so a host that uses the old names works.

In [ ]:
#| export
STATIC = PKG/'static'
LIBS = ('rough.js', 'perfect-freehand.mjs')

def assets() -> AttrDict:
    "The files of the board: `js` and `css` in bundle order, `tex` (the formula parts), and `libs`, the folder of the pen libraries and their licences."
    return AttrDict(js=[STATIC/'texparts.js', STATIC/'board.js'], css=[STATIC/'paper.css', STATIC/'board.css'], tex=STATIC/'texparts.js',
                    libs=STATIC/'vendor'/'board')

def bundle(kind:str='js') -> str:
    "The board as one script (`js`) or one style sheet (`css`)."
    return '\n'.join(f'/* rupaka/{p.relative_to(PKG).as_posix()} */\n{p.read_text()}' for p in assets()[kind])

Each file is in the package, and the script names the new global and the old one:

In [ ]:
a = assets()
assert all(p.exists() for p in [*a.js, *a.css, a.tex]) and all((a.libs/f).exists() for f in (*LIBS, 'LICENSES.md'))
js = bundle()
assert 'window.rupakaBoard = window.abhyasaBoard' in js and 'window.rupakaTex = window.abhyasaTex' in js
assert '--rk-paper:' in bundle('css') and '.board-panel' in bundle('css')

### The board language

A reply is JSON lines: one object on each line. The server checks each line alone, so a bad line drops and the rest
draws. There are 14 commands. The model never writes HTML, CSS, script or a URL.

| op | fields | what it does |
|---|---|---|
| `step` | `say`, and `src` with `quote` | starts a step; `say` is the spoken line. A quote is word for word from source `src` |
| `circle`, `box`, `underline` | `at` | a hand-drawn ring, box or line on the anchor |
| `arrow` | `from`, `to` | a curved arrow between two anchors |
| `label` | `at`, `text`, `side` | a short note beside the anchor |
| `sketch` | `at`, `pts` | a free line in the box of a figure anchor, to mark a part of the figure |
| `point` | `at` | moves a pointer mark to the anchor |
| `clear` | `n` (optional) | removes the marks of all steps, or of step `n` |
| `demo_set` | `set` | changes inputs of the step's rupaka demo, only after its gate is open |
| `video_at` | `t`, `quote` | moves the explainer video to `t` seconds, and quotes its transcript there |
| `panel` | `title`, `head`, `cols`, `numbered` (each optional) | starts the side panel at the top right of the window: a small title, a bold heading and the names of two columns. A new `panel` starts a new beat |
| `row` | `cells`, and `to`, `at`, `chain` (optional) | adds one row to the side panel: the cells as boxes, then an arrow and `to`. With `at`, the page circles that thing and a hand-drawn curve links it to the row. With `chain`, the cells are words with arrows between them |
| `note` | `text` | one quiet line in the side panel |

Each stroke can have `w`: the word of `say` where it starts.

A `sketch` marks a part of a figure only, so no shape covers the words of the page. A picture of an idea goes in the side panel.
For a formula, the tutor gives an example from day-to-day life and marks the page's own formula.

The side panel is the place for a small concrete example (**Explain**). The tutor builds it one row in each step, from the
real values on the page: its row names the anchor of the value in `at`. The page circles the value and draws a curve from it to
the row, so the model writes no circle for it. A process of 3 to 5 steps is a panel with `numbered` rows, or one `chain` row.
A new `panel` line starts the next beat, for example from the values on the page to the big idea. The side panel is the one
picture of Explain. Show is a rupaka demo.

In [ ]:
#| export
OPS = ('step', 'circle', 'box', 'underline', 'arrow', 'label', 'sketch', 'point', 'clear', 'demo_set', 'video_at', 'panel', 'row', 'note')
FIELDS = dict(step={'say', 'src', 'quote'}, circle={'at'}, box={'at'}, underline={'at'}, arrow={'from', 'to'},
              label={'at', 'text', 'side'}, sketch={'at', 'pts'}, point={'at'}, clear={'n'}, demo_set={'set'}, video_at={'t', 'quote'},
              panel={'title', 'head', 'cols', 'numbered'}, row={'cells', 'to', 'at', 'chain'}, note={'text'})
LIMITS = dict(steps=6, strokes=6, lines=60, line_bytes=600, pts=64, say=160, text=60, quote_min=40, quote_max=600, set=6,
              step_bytes=1500, cells=5, cell=24, title=40, head=60, note=90, rows=6)
PANEL = ('panel', 'row', 'note')   # the lines of the side panel
MARKS = ('circle', 'box', 'underline', 'label')   # a mark on a thing: it needs a thing under it, so never a bare point
META = re.compile(r"\b(?:my (?:earlier|last|first|previous) (?:offer|answer|board|reply|step)|(?:was|is) misleading|i apologi[sz]e|sorry|i was wrong"
                  r"|no (?:live )?demo (?:that )?i can|(?:cannot|can't|can not) (?:start|show) (?:a|the|any) demo)", re.I)   # a line about the board, not the idea
META_WHY = 'a line about the board, not about the idea'
SIDES = ('above', 'below', 'left', 'right')
SURE = 0.7   # the least `sure` of a board whose steps go out as they come (`stream_board`)

The first line can be a header: `{"op":"v","n":1,"sure":0.9,"need":"none"}`. `sure` is how sure the model is, from 0 to
1. `need` says what it needs to be sure: `none`, `vault` or `web`.

In [ ]:
#| export
NEEDS = ('none', 'vault', 'web')
VERSION = 1

### Anchors

A stroke names a thing on the page. The page sends a list of anchors, each with an `id`, a `kind` and its `text`. A
formula also has its `tokens` (the TeX of its parts), a figure has its `labels` (a box for each label in the image),
and the demo has its inputs.

| form | example | the place |
|---|---|---|
| an id | `"passage"` | the whole anchor |
| words | `{"a": "passage", "text": "closed form"}` | the words in the anchor |
| a maths token | `{"a": "m1", "tex": "\\bar\\alpha_t"}` | the part of the formula |
| a figure label | `{"a": "figure", "label": "a"}` | the label in the image |
| a box | `{"a": "figure", "box": [100, 200, 400, 600]}` | a part of the anchor, from 0 to 1000 |
| a point | `{"a": "m1", "pt": [1100, 500]}` | a point beside the anchor, for the far end of an arrow |

Two ways to write the same TeX must match. `norm_tex` takes out the spaces and the braces around one symbol.

In [ ]:
#| export
_ONE = re.compile(r'\{(\\[A-Za-z]+|[^{}\\])\}')

def norm_tex(s:str) -> str:
    "`s` with no white space, and with no braces around one character or one command, so `\\bar{\\alpha}_{t}` is `\\bar\\alpha_t`."
    s = re.sub(r'\s+', '', str(s or ''))
    while (t := _ONE.sub(r'\1', s)) != s: s = t
    return s

Braces and spaces do not change the match:

In [ ]:
test_eq(norm_tex(r'\bar{\alpha}_{t}'), norm_tex(r'\bar\alpha_t'))
test_eq(norm_tex(r'\sqrt{\bar\alpha_t}'), r'\sqrt{\bar\alpha_t}')   # a group of two symbols keeps its braces
test_ne(norm_tex(r'\alpha_t'), norm_tex(r'\bar\alpha_t'))

Words match with no case and with white space made single, as the page shows them.

In [ ]:
#| export
def flat(s) -> str: return ' '.join(str(s or '').split()).lower()

def _box(b, lo=0, hi=1000) -> bool:
    "Whether `b` is `[x0, y0, x1, y1]` with numbers from `lo` to `hi`, and `x0 < x1`, `y0 < y1`."
    return (isinstance(b, list) and len(b) == 4 and all(_num(x) and lo <= x <= hi for x in b) and b[0] < b[2] and b[1] < b[3])

def _num(x) -> bool: return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)

`anchor_why` gives the reason that an anchor is not on the page, or `''` when it is. The context `ctx` has the anchors
by id in `ctx.by`.

In [ ]:
#| export
FORMS = ('text', 'tex', 'label', 'box', 'pt')

def _edge(w:str, end:bool) -> str:
    "A word boundary at the start (or `end`) of `w`, when `w` starts (or ends) with a letter or a digit; else none."
    c = w[-1] if end else w[0]
    return '' if not (c.isalnum() or c == '_') else r'(?!\w)' if end else r'(?<!\w)'

def _text_why(v, a) -> str:
    """Why the words `v` cannot mark the anchor `a`, or ''. They must be in its text once, as whole words, and not inside a
    formula: the page writes a formula as its TeX between $ signs, and draws it as maths, so the words are not on the page."""
    if not (isinstance(v, str) and v.strip()): return f"no words {v!r} in {a['id']}"
    w, t = flat(v), flat(a.get('text'))
    ms = [m.span() for m in re.finditer(_edge(w, False) + re.escape(w) + _edge(w, True), t)]
    if not ms: return f"no words {v!r} in {a['id']}"
    if len(ms) > 1: return f"the words {v!r} are in {a['id']} more than once"
    s, e = ms[0]
    if a.get('kind') != 'code' and any(s < f1 and f0 < e for f0, f1 in (m.span() for m in re.finditer(r'\$[^$]+\$', t))):
        return f"words inside a formula of {a['id']}: mark its part with tex"
    return ''

def anchor_why(at, ctx) -> str:
    "Why `at` names no anchor of `ctx`, or '' when it does."
    if isinstance(at, str): return '' if at in ctx.by else f'no anchor {at!r}'
    if not isinstance(at, dict) or set(at) - {'a', *FORMS} or not isinstance(at.get('a'), str): return 'a bad anchor'
    if (a := ctx.by.get(at['a'])) is None: return f"no anchor {at['a']!r}"
    forms = [f for f in FORMS if f in at]
    if len(forms) > 1: return 'an anchor with two forms'
    if not forms: return ''
    f, v = forms[0], at[forms[0]]
    if f == 'text': return _text_why(v, a)
    if f == 'tex': return '' if isinstance(v, str) and norm_tex(v) in {norm_tex(t) for t in a.get('tokens') or ()} else f'no token {v!r} in {at["a"]}'
    if f == 'label': return '' if isinstance(v, str) and v in (a.get('labels') or {}) else f'no label {v!r} in {at["a"]}'
    if f == 'box': return '' if _box(v) else 'a bad box'
    return '' if isinstance(v, list) and len(v) == 2 and all(_num(x) and -1000 <= x <= 2000 for x in v) else 'a bad point'

A small context for the tests: a passage, the DDPM formula with its tokens, a triangle figure with labels `a` and `b`,
and a demo with the input `t`.

In [ ]:
#| export
def by_id(ctx) -> AttrDict:
    "`ctx` with `by`: its anchors by id."
    ctx = AttrDict(ctx)
    ctx.by = {a['id']: a for a in ctx.get('anchors') or ()}
    return ctx

In [ ]:
DDPM = r'q(\mathbf{x}_t\mid\mathbf{x}_0)=\mathcal{N}(\mathbf{x}_t;\sqrt{\bar\alpha_t}\mathbf{x}_0,(1-\bar\alpha_t)\mathbf{I})'
ctx = by_id(dict(anchors=[dict(id='passage', kind='passage', text='It admits sampling x_t at an arbitrary timestep t in closed form.'),
                          dict(id='m1', kind='formula', text=DDPM, tokens=[r'\bar\alpha_t', r'\sqrt{\bar\alpha_t}', r'\mathbf{x}_0', r'(1-\bar\alpha_t)']),
                          dict(id='out', kind='text', text='tensor([18]) gives 47. 1.0 and 1.0 give 1.414, and 2.0 gives 2.0. Guess, compare, then update the weights.'),
                          dict(id='figure', kind='figure', text='A right triangle', labels=dict(a=[100, 420, 160, 500], b=[480, 880, 540, 960]))],
                 demo=dict(inputs=dict(t=dict(value=500, min=0, max=1000, label='Step t')), open=False)))

Each form finds its anchor, and a wrong name does not:

In [ ]:
test_eq(anchor_why('passage', ctx), '')
test_eq(anchor_why(dict(a='m1', tex=r'\bar{\alpha}_t'), ctx), '')
test_eq(anchor_why(dict(a='passage', text='Closed  form'), ctx), '')
test_eq(anchor_why(dict(a='figure', label='a'), ctx), '')
test_eq(anchor_why(dict(a='figure', box=[0, 0, 500, 500]), ctx), '')
assert anchor_why('e7', ctx)                                  # an id that the page did not send
assert anchor_why(dict(a='m1', tex=r'\beta_t'), ctx)          # a token that is not in the formula
assert anchor_why(dict(a='passage', text='open form'), ctx)   # words that are not in the passage
assert anchor_why(dict(a='figure', label='c'), ctx)           # a label that is not in the figure
assert anchor_why(dict(a='figure', box=[500, 0, 100, 10]), ctx)
assert anchor_why(dict(a='m1', tex=r'\bar\alpha_t', text='x'), ctx)   # two forms at once

Words mark a thing only when the page shows them in one place. A part of a word does not count, so "form" does not mark the
"form" of "transform". Words that are in the anchor two times drop, because the page cannot know which place the model means. Words
inside a formula drop too: the page draws the formula as maths, so a mark goes on its part with `tex`.

In [ ]:
tctx = by_id(dict(anchors=[dict(id='p', kind='passage', text='A transform maps symbol representations $(x_1, \\dots, x_n)$ to continuous representations $z$.')]))
test_eq(anchor_why(dict(a='p', text='continuous representations'), tctx), '')
test_eq(anchor_why(dict(a='p', text='form'), tctx), "no words 'form' in p")
test_eq(anchor_why(dict(a='p', text='representations'), tctx), "the words 'representations' are in p more than once")
test_eq(anchor_why(dict(a='p', text='x_1'), tctx), 'words inside a formula of p: mark its part with tex')
test_eq(anchor_why(dict(a='p', text='$z$'), tctx), 'words inside a formula of p: mark its part with tex')

### The check of one line

`check_line` gives the reason that a line drops, or `''`. Each string in a line is checked too: no tag, no script and no
link can get in, also as the text of a label.

In [ ]:
#| export
UNSAFE = re.compile(r'<[a-z/!?]|javascript:|https?://|www\.|data:', re.I)

def _strings(o):
    "Every string inside `o`, also the keys."
    if isinstance(o, str): yield o
    elif isinstance(o, dict):
        for k, v in o.items(): yield k; yield from _strings(v)
    elif isinstance(o, list):
        for v in o: yield from _strings(v)

A cue of the explainer video is `[t, text]`. `cue_at` gives the cue that plays at `t` seconds. `cue_quote` finds a quote in
the cue at `t` or in the cue before or after it. It gives the start of the cue where the quote begins, and the quote as the
transcript says it. The board seeks the video to that start, so a time that the model gives a little late still lands on
the right cue.

In [ ]:
#| export
def cue_at(cues, t:float):
    "The cue `(start, text)` that plays at `t`: the last one that starts at or before `t`. None when there is none."
    out = None
    for c in cues or ():
        if c[0] <= t: out = c
        else: break
    return out

def cue_quote(cues, t:float, quote:str):
    "`(start, words)` when `quote` (3 words or more) is word for word in the cue at `t` with the cue before and after it: the start of the cue where it begins, and the quote as the transcript says it. Else None."
    if not isinstance(quote, str) or len(quote.split()) < 3 or not (c := cue_at(cues, t)): return None
    i = list(cues).index(c)
    near = cues[max(0, i - 1):i + 2]
    parts = [' '.join(str(x[1]).split()) for x in near]
    m = re.search(rf'(?<!\w){re.escape(" ".join(quote.split()))}(?!\w)', ' '.join(parts), re.I)
    if not m: return None
    at = 0
    for k, p in enumerate(parts):
        if m.start() < at + len(p) + 1: return near[k][0], m.group(0)
        at += len(p) + 1

def _in_cues(cues, t:float, quote:str) -> bool: return cue_quote(cues, t, quote) is not None

In [ ]:
#| export
def _demo_why(o, ctx) -> str:
    d = ctx.get('demo') or {}
    ins, s = d.get('inputs') or {}, o.get('set')
    if not d.get('open'): return 'the demo gate is closed'
    if not isinstance(s, dict) or not 1 <= len(s) <= LIMITS['set']: return 'a bad set'
    for k, v in s.items():
        if k not in ins: return f'no demo input {k!r}'
        if not isinstance(v, (int, float, str, bool)) or (isinstance(v, str) and len(v) > 60): return f'a bad value for {k!r}'
        lo, hi = ins[k].get('min'), ins[k].get('max')
        if _num(v) and ((_num(lo) and v < lo) or (_num(hi) and v > hi)): return f'{k} out of range'
    return ''

def _short_ok(t, n:int) -> bool: return isinstance(t, str) and bool(t.strip()) and len(t) <= n and t.isprintable()

def _kind(at, ctx) -> str:
    "The kind of the anchor that `at` names (`at` passed `anchor_why`)."
    return (ctx.by.get(at if isinstance(at, str) else at['a']) or {}).get('kind', '')

def plain_say(say:str) -> str:
    "The spoken line `say` with no semicolon: each part is a sentence of its own."
    return re.sub(r'\s*;\s*(\w?)', lambda m: '. ' + m[1].upper(), ' '.join(say.split())).strip()

def _cell(c, n:int) -> bool:
    "Whether `c` is a number or a short text. The `$` signs around maths do not count."
    return (_num(c) and not isinstance(c, bool)) or (isinstance(c, str) and _short_ok(c, n + c.count('$')))

def _panel_why(o, ctx) -> str:
    "Why the side-panel line `o` (`panel`, `row` or `note`) drops, or ''."
    if o['op'] == 'panel':
        for k in ('title', 'head'):
            if k in o and not _short_ok(o[k], LIMITS[k]): return f'a bad {k}'
        cs = o.get('cols', [])
        if not (isinstance(cs, list) and len(cs) <= 2 and all(_short_ok(c, LIMITS['cell'] + 8) for c in cs)): return 'bad column names'
        return '' if isinstance(o.get('numbered', False), bool) else 'a bad numbered'
    if o['op'] == 'note': return '' if _short_ok(o.get('text'), LIMITS['note']) else 'a bad note'
    cs = o.get('cells')
    if not (isinstance(cs, list) and 1 <= len(cs) <= LIMITS['cells'] and all(_cell(c, LIMITS['cell']) for c in cs)): return 'bad cells'
    if 'to' in o and not _cell(o['to'], LIMITS['cell']): return 'a bad to'
    if not isinstance(o.get('chain', False), bool): return 'a bad chain'
    return anchor_why(o['at'], ctx) if 'at' in o else ''

def check_line(o, ctx) -> str:
    "Why the reply line `o` drops, or '' when it is good. `ctx` comes from `by_id`."
    if not isinstance(o, dict) or o.get('op') not in OPS: return 'an unknown op'
    op = o['op']
    if (bad := set(o) - {'op', 'w', 'n'} - FIELDS[op]): return f'unknown keys {sorted(bad)}'
    if any(UNSAFE.search(s) for s in _strings(o)): return 'a tag, a script or a link'
    if 'w' in o and not (isinstance(o['w'], int) and not isinstance(o['w'], bool) and 0 <= o['w'] < 100): return 'a bad word mark'
    if op == 'step':
        say = o.get('say')
        if not (isinstance(say, str) and say.strip() and len(say) <= LIMITS['say'] and say.isprintable()): return 'a bad say'
        return META_WHY if META.search(say) else ''
    if op in PANEL: return _panel_why(o, ctx)
    if op in MARKS and isinstance(o.get('at'), dict) and 'pt' in o['at']: return 'a mark on a bare point, with no thing under it'
    if op in ('circle', 'box', 'underline', 'point', 'label', 'sketch'):
        if (why := anchor_why(o.get('at'), ctx)): return why
    if op == 'sketch' and _kind(o['at'], ctx) != 'figure': return 'a sketch that is not on a figure'
    if op == 'arrow': return anchor_why(o.get('from'), ctx) or anchor_why(o.get('to'), ctx)
    if op == 'label':
        t = o.get('text')
        if not (isinstance(t, str) and t.strip() and len(t) <= LIMITS['text'] and t.isprintable()): return 'a bad label text'
        if o.get('side', 'above') not in SIDES: return 'a bad side'
    if op == 'sketch':
        p = o.get('pts')
        if not (isinstance(p, list) and 2 <= len(p) <= LIMITS['pts']): return 'a bad sketch'
        if not all(isinstance(q, list) and len(q) == 2 and all(_num(x) and -1000 <= x <= 2000 for x in q) for q in p): return 'a bad sketch point'
    if op == 'clear' and 'n' in o and not (isinstance(o['n'], int) and 1 <= o['n'] <= LIMITS['steps']): return 'a bad step number'
    if op == 'demo_set': return _demo_why(o, ctx)
    if op == 'video_at':
        v, t, q = ctx.get('video') or {}, o.get('t'), o.get('quote')
        if not v.get('cues') or not re.fullmatch(r'[A-Za-z0-9_-]{11}', str(v.get('id') or '')): return 'no video on the page'
        if not (_num(t) and t >= 0 and isinstance(q, str) and len(q) <= 200): return 'a bad video mark'
        if not _in_cues(v['cues'], t, q): return 'the quote is not in the transcript at that time'
    return ''

Good lines pass. The checks drop an unknown op, an unknown key, a link and markup:

In [ ]:
test_eq(check_line(dict(op='circle', at=dict(a='m1', tex=r'\bar\alpha_t'), w=0), ctx), '')
test_eq(check_line(dict(op='label', at=dict(a='figure', label='a'), text='side a', side='left'), ctx), '')
test_eq(check_line(dict(op='arrow', **{'from': dict(a='m1', pt=[1100, 500]), 'to': dict(a='m1', tex=r'(1-\bar\alpha_t)')}), ctx), '')
test_eq(check_line(dict(op='clear'), ctx), '')
assert check_line(dict(op='script', code='alert(1)'), ctx)
assert check_line(dict(op='circle', at='passage', html='<b>'), ctx)
assert check_line(dict(op='label', at='passage', text='see https://evil.example'), ctx)
assert check_line(dict(op='label', at='passage', text='<img src=x onerror=alert(1)>'), ctx)
assert check_line(dict(op='step', say='x' * 161), ctx)

A sketch marks a part of a figure, with at most 64 points. A sketch on another anchor drops: a picture of an idea goes in the side panel.

In [ ]:
test_eq(check_line(dict(op='sketch', at=dict(a='figure', pt=[1050, 0]), pts=[[0, 0], [1000, 1000]]), ctx), '')
assert check_line(dict(op='sketch', at='figure', pts=[[i, i] for i in range(65)]), ctx)
test_eq(check_line(dict(op='sketch', at=dict(a='m1', pt=[1050, 0]), pts=[[0, 0], [1000, 1000]]), ctx), 'a sketch that is not on a figure')

A circle, a box, an underline or a label marks a thing on the page. On a bare point it has no thing under it, so it drops. A point and an arrow can use a point:

In [ ]:
assert 'bare point' in check_line(dict(op='box', at=dict(a='passage', pt=[-200, 900])), ctx)
assert 'bare point' in check_line(dict(op='circle', at=dict(a='passage', pt=[500, 500])), ctx)
test_eq((check_line(dict(op='point', at=dict(a='passage', pt=[-200, 900])), ctx), check_line(dict(op='box', at=dict(a='passage', box=[0, 0, 500, 500])), ctx)), ('', ''))

The side panel: a `panel` line with its title, a `row` of short cells with an optional `to` and anchor, and a `note`. A number in a cell is good. A long cell, an empty row or an unknown anchor drops:

In [ ]:
test_eq([check_line(o, ctx) for o in (dict(op='panel', title='tensors and targets', head='Make a question and its answer', cols=['Input', 'Target']),
                                       dict(op='row', cells=['[18]'], to=47, at=dict(a='passage', text='closed form')),
                                       dict(op='row', cells=['Guess', 'Compare', 'Adjust'], chain=True), dict(op='note', text='Both can be tensors.'))], [''] * 4)
for bad in (dict(op='row', cells=[]), dict(op='row', cells=['x' * 25]), dict(op='row', cells=['a'], at='e7'), dict(op='panel', cols=['a', 'b', 'c']),
            dict(op='note', text=''), dict(op='row', cells=['a'], chain='yes')): assert check_line(bad, ctx), bad

The demo gate: `demo_set` drops while the learner has not locked in a prediction, so the board cannot give the result
first. After the lock-in it changes only inputs that the demo has, in their range.

In [ ]:
assert 'gate' in check_line(dict(op='demo_set', set=dict(t=500)), ctx)
open_ctx = by_id({**ctx, 'demo': {**ctx.demo, 'open': True}})
test_eq(check_line(dict(op='demo_set', set=dict(t=500)), open_ctx), '')
assert check_line(dict(op='demo_set', set=dict(beta=1)), open_ctx)
assert check_line(dict(op='demo_set', set=dict(t=5000)), open_ctx)

`video_at` quotes the transcript at its time, word for word:

In [ ]:
vctx = by_id({**ctx, 'video': dict(id='abc123def45', cues=[[0, 'Today we add noise.'], [61.5, 'Alpha bar is the share of the signal that is left.'], [70, 'It falls as t grows.']])})
test_eq(check_line(dict(op='video_at', t=63, quote='the share of the signal'), vctx), '')
assert check_line(dict(op='video_at', t=63, quote='the share of the noise'), vctx)
assert check_line(dict(op='video_at', t=63, quote='Alpha bar'), vctx)      # too short to be a quote
assert check_line(dict(op='video_at', t=63, quote='the share of the signal'), ctx)   # no video on the page

`cue_quote` gives the start of the cue that holds the quote, and the words as the transcript has them:

In [ ]:
cs = vctx.video['cues']
test_eq(cue_quote(cs, 63, 'THE SHARE of the signal'), (61.5, 'the share of the signal'))
test_eq(cue_quote(cs, 71, 'share of the signal'), (61.5, 'share of the signal'))   # the time is late: the cue before holds it
test_eq(cue_quote(cs, 63, 'left. It falls'), (61.5, 'left. It falls'))             # a quote across two cues starts in the first
test_eq(cue_quote(cs, 5, 'the share of the signal'), (61.5, 'the share of the signal'))
assert cue_quote(cs, 200, 'Today we add noise.') is None                           # too far from its time

### The whole reply

`parse_board` reads a reply line by line. Each step keeps at most 6 strokes, and a board has at most 6 steps. A stroke
before the first step drops. Two dropped lines in a row end the board (`failed`): the steps before them stay.

A step can quote a source: `src` is the number of the source in `ctx.sources`, and `quote` must be word for word in its
text, with 40 to 600 characters. `verbatim` checks it: white space is made single, and the quote goes from a word boundary to
a word boundary. A host can give its own check as `verbatim(quote, text) -> str` (abhyasa gives the rules of its sources).
A quote that fails drops, and the step stays, in the tutor's words.

### Each row with its step

A step says one thing, and its row and its marks show that same thing. So the spoken line of a step must name what its row
shows. `key_terms` gives the key words and symbols of a text: each word of 3 letters or more that is not a small word, each
number, and each maths token such as `x_i`. `named` is true when a row (or a note) and a spoken line share one of them. The
check is cheap and knows no subject. `page_terms` gives the key terms of what the page shows.

`parse_board` drops a row that names nothing on the page, because the panel is an example made from the page. Then it
pairs each row and each note with its step (`_pair`). A row that its own step does not name moves to the next step
of its panel that names it. A row that no step names drops. A row never moves back to an earlier step, so a
stream sends each row once, with the step that explains it.

In [ ]:
#| export
SMALL = frozenset('the and but for nor yet with from into onto than then that this these those there here what which who whom whose when where why how '
                  'its it’s are was were been being has have had does did not each every one any all some most more less only also just very '
                  'can could will would shall should may might must you your our their they them she her him his out over under about '
                  'same other next last first own such like get gets got make makes made use uses used see sees shows show step page'.split())
_TERM = re.compile(r'\\[A-Za-z]+|[A-Za-z]+(?:[_^](?:\{[^{}]{1,12}\}|[A-Za-z0-9]+))+|[A-Za-z]+|\d+(?:\.\d+)?')

def key_terms(s) -> set:
    """The key words and symbols of `s`: each word of 3 letters or more that is not a small word, cut to its first 5 letters; each
    number; each maths token such as `x_i` or `w_1x_1`; each single letter other than "a" and "I". `\\sum` counts as the word sum."""
    out = set()
    for t in _TERM.findall(str(s or '').replace('$', ' ')):
        if t[0] == '\\': t = t[1:]
        if t[0].isdigit() or '_' in t or '^' in t: out.add(norm_tex(t)); continue
        if len(t) == 1: out.update({t} - {'a', 'A', 'I'}); continue
        if len(t) >= 3 and (w := t.lower()) not in SMALL: out.add(w[:5])
    return out

def _same(a:str, b:str) -> bool:
    "Whether two key terms are the same: the same symbol or number, or two words where one is the start of the other (add, adds, adding)."
    if a == b: return True
    return a.isalpha() and b.isalpha() and len(a) > 1 and len(b) > 1 and (a.startswith(b) or b.startswith(a))

def _line_terms(o) -> set: return key_terms(' '.join(str(x) for x in [*o.get('cells', ()), o.get('to', ''), o.get('text', '')]))
def _shares(ts, us) -> bool: return any(_same(t, u) for t in ts for u in us)

def named(o, say:str) -> bool:
    "Whether the spoken line `say` names the panel line `o` (a row or a note): they share a key word or a symbol."
    return _shares(_line_terms(o), key_terms(say))

def page_terms(ctx) -> set:
    "The key terms of what the page shows: the text, tokens and labels of each anchor, the sources, and the demo."
    d = ctx.get('demo') or {}
    parts = [x for a in ctx.get('anchors') or () for x in (a.get('text', ''), *(a.get('tokens') or ()), *(a.get('labels') or ()))]
    parts += [s.get('text') or '' for s in ctx.get('sources') or ()] + list(d.get('readouts') or ()) + [str(v.get('label', '')) for v in (d.get('inputs') or {}).values()]
    return key_terms(' '.join(map(str, parts)))

def _pair(out, ctx):
    """A row that names nothing on the page drops: the panel is an example made from the page, not from a later idea.
    Then each row and note goes with the step whose spoken line names it. A row that its own step does not name moves to the next step of
    its beat that names it; else it drops. It never moves back, so a stream sends it once, with its step. A circle line on its thing stays
    where it is: the stream sent it already, and the row draws its own circle."""
    page = page_terms(ctx)
    for st in out.steps:
        if page: st['strokes'] = [s for s in st['strokes'] if s['op'] != 'row' or _shares(_line_terms(s), page) or out.dropped.append((-1, 'a row that names nothing on the page'))]
    for k, st in enumerate(out.steps):
        keep, moves = [], {}
        beat = list(takewhile(lambda t: not any(x['op'] == 'panel' for x in t['strokes']), out.steps[k + 1:]))
        for s in st['strokes']:
            if s['op'] not in PANEL[1:] or named(s, st['say']): keep.append(s); continue
            room = lambda t: len(t['strokes']) + len(moves.get(id(t), ())) < LIMITS['strokes']
            if (to := next((t for t in beat if named(s, t['say']) and room(t)), None)) is None:
                out.dropped.append((-1, f"a {s['op']} that its step does not name")); continue
            moves.setdefault(id(to), []).append({f: v for f, v in s.items() if f != 'w'})   # its word mark was a word of the other line
        st['strokes'] = keep
        for t in beat:
            if (ms := moves.get(id(t))):   # before the rows of that step: they came first in the reply
                at = next((i for i, x in enumerate(t['strokes']) if x['op'] in PANEL), len(t['strokes']))
                t['strokes'][at:at] = ms

### One shape for each panel

A panel is a small table or a small flow, and its rows line up only when they have one shape. A row is a *pair* (one or
more cells, then an arrow and the answer `to`) or a *chain* (words with an arrow between each two). The first row of a
panel sets its shape, and `_shape` makes the other rows keep it:

- A pair with an answer goes into a chain panel as a chain of its cells and its answer: it loses nothing.
- A chain of two words goes into a pair panel as one cell and its answer: it loses nothing too.
- Any other row of another shape drops, and so does a pair with another count of cells.
- The column names `cols` stand over the cells and the answer of a pair. They stay only when the first row is a pair with one
  cell less than the names and an answer. Else they go, and the rows stay: two short names say less than the rows.
- A panel shows at most 6 rows. More do not fit next to the page.

The first row decides, and not the most rows, because a stream sends each row with its step: a later row cannot change the
rows that show. For the same reason a stream holds a panel line with column names until its first row is in (`_held`).

In [ ]:
#| export
def _form(r) -> tuple:
    "The shape of the row `r`: `('chain', parts)` or `('pair', cells, whether it has an answer)`."
    return ('chain', len(r['cells'])) if r.get('chain') else ('pair', len(r['cells']), 'to' in r)

def _fit(r, first):
    "The row `r` in the shape of the `first` row of its panel, or None when it cannot take that shape and lose nothing."
    f, g = _form(first), _form(r)
    if f[0] == 'chain':
        if g[0] == 'chain': return r
        return {**{k: v for k, v in r.items() if k != 'to'}, 'cells': [*r['cells'], r['to']], 'chain': True} if g[2] else None
    if g == ('chain', 2): r, g = {**{k: v for k, v in r.items() if k != 'chain'}, 'cells': r['cells'][:1], 'to': r['cells'][1]}, ('pair', 1, True)
    return r if g == f else None

def _shape(out):
    "Each panel of the board `out` with the shape of its first row, column names only over the pairs that they fit, and at most 6 rows."
    panel, first, n = None, None, 0
    for st in out.steps:
        keep = []
        for s in st['strokes']:
            if s['op'] == 'panel': panel, first, n = s, None, 0
            if s['op'] != 'row': keep.append(s); continue
            if first is None:
                first = s
                if panel and 'cols' in panel and _form(s) != ('pair', len(panel['cols']) - 1, True):
                    del panel['cols']; out.dropped.append((-1, 'column names that do not fit the first row'))
            elif (s := _fit(s, first)) is None: out.dropped.append((-1, 'a row of another shape than the first row of its panel')); continue
            if n >= LIMITS['rows']: out.dropped.append((-1, f"more than {LIMITS['rows']} rows in a panel")); continue
            n += 1; keep.append(s)
        st['strokes'] = keep

In [ ]:
#| export
WATCH = 'https://www.youtube.com/watch?v='

def yt_link(vid:str, t:float) -> str:
    "The watch link that opens the video `vid` at `t` seconds, as fossick's `yt_link` writes it."
    return f'{WATCH}{vid}&t={int(t)}s'

def verbatim(quote:str, text:str) -> str:
    "`quote` with its white space made single, when it is in `text` word for word, from a word boundary to a word boundary, and has 40 to 600 characters; else ''."
    q, t = ' '.join(str(quote or '').split()), ' '.join(str(text or '').split())
    if not LIMITS['quote_min'] <= len(q) <= LIMITS['quote_max']: return ''
    return q if re.search(rf'(?<!\w){re.escape(q)}(?!\w)', t) else ''

def _quote(o, ctx, verbatim=verbatim):
    "The source and the word-for-word quote of step line `o`, or None."
    src, q, ss = o.get('src'), o.get('quote'), ctx.get('sources') or []
    if not (isinstance(src, int) and 1 <= src <= len(ss) and isinstance(q, str)): return None
    if not (v := verbatim(q, ss[src - 1].get('text') or '')): return None
    return dict(src=src, text=v)

def _video_mark(o, ctx) -> dict:
    "The checked `video_at` line `o`, at the start of the cue that holds its quote, with the quote as the transcript says it and the `yt_link` to that time."
    t, q = cue_quote(ctx.video['cues'], o['t'], o['quote'])
    return dict(op='video_at', t=t, quote=q, link=yt_link(ctx.video['id'], t), **({'w': o['w']} if 'w' in o else {}))

def _stroke(o, ctx) -> dict:
    "The checked stroke line `o` as the page gets it."
    if o['op'] == 'video_at': return _video_mark(o, ctx)
    if o['op'] == 'row':
        cs, to = [str(c) for c in o['cells']], o.get('to')
        if to is None and not o.get('chain') and len(cs) >= 2: cs, to = cs[:-1], cs[-1]   # the answer in a last cell: it goes after the arrow
        return {**{k: v for k, v in o.items() if k not in ('n', 'to')}, 'cells': cs, **({'to': str(to)} if to is not None else {})}
    return {k: v for k, v in o.items() if k != 'n' or o['op'] == 'clear'}

def _has_strokes(o) -> bool:
    "Whether `o` is a step with its strokes inside it: such a line can have up to 1500 bytes."
    return isinstance(o, dict) and o.get('op') == 'step' and isinstance(o.get('strokes'), list)

def _objects(lines):
    """Each line as one object, or None when it is not one JSON object of 600 bytes or less (1500 for a step with its strokes inside it).
    A step with its strokes inside it gives the step, then each stroke."""
    for line in lines:
        n = len(line.encode())
        try: o = json.loads(line) if n <= LIMITS['step_bytes'] else None
        except ValueError: o = None
        if n > LIMITS['line_bytes'] and not _has_strokes(o): o = None
        if isinstance(o, dict) and o.get('op') == 'step' and isinstance(o.get('strokes'), list):
            ss = o.pop('strokes')
            yield o
            yield from (s if isinstance(s, dict) else None for s in ss[:LIMITS['strokes'] + 1])
        else: yield o

NUMBERS = dict(two=2, three=3, four=4, five=5, six=6, seven=7, eight=8, nine=9, ten=10)
_COUNT = re.compile(r'\b(two|three|four|five|six|seven|eight|nine|ten|[2-9])\s+', re.I)

def _counts(steps):
    "A head that names a count of parts (\"three stages\") that its panel does not have loses the count."
    ss = [s for st in steps for s in st['strokes'] if s['op'] in PANEL]
    for k, s in enumerate(ss):
        if s['op'] != 'panel' or not (m := _COUNT.search(s.get('head', ''))): continue
        n = sum(1 for x in takewhile(lambda x: x['op'] != 'panel', ss[k + 1:]) if x['op'] == 'row')
        if NUMBERS.get(m[1].lower(), int(m[1]) if m[1].isdigit() else 0) != n: s['head'] = _COUNT.sub('', s['head'], 1)

def parse_board(text:str, ctx, verbatim=verbatim) -> AttrDict:
    "The checked board in the model's reply `text`: `steps`, `dropped` lines with why, `sure`, `need` and `failed`. `verbatim` checks a quote."
    out = AttrDict(steps=[], dropped=[], sure=None, need='none', failed=False)
    run, skip = 0, False   # `skip`: the step before was about the board, so its strokes go too
    lines = [l.strip() for l in str(text or '').splitlines() if l.strip() and not l.strip().startswith('```')]
    for k, o in enumerate(_objects(lines[:LIMITS['lines']])):
        why = ''
        if o is None: why = 'not one JSON object of 600 bytes or less'
        elif isinstance(o, dict) and o.get('op') == 'v' and k == 0:
            if o.get('n', VERSION) > VERSION: out.failed = True; out.dropped.append((k, 'a newer board version')); break
            if _num(o.get('sure')): out.sure = max(0., min(1., float(o['sure'])))
            if o.get('need') in NEEDS: out.need = o['need']
            continue
        elif isinstance(o, dict) and o.get('op') != 'step' and skip: out.dropped.append((k, 'a stroke of a dropped step')); continue
        elif not (why := check_line(o, ctx)):
            if o['op'] == 'step':
                if len(out.steps) >= LIMITS['steps']: why = 'more than 6 steps'
                else:
                    s = dict(n=len(out.steps) + 1, say=plain_say(o['say']), strokes=[])
                    if (q := _quote(o, ctx, verbatim)): s['quote'] = q
                    elif 'quote' in o: out.dropped.append((k, 'the quote is not word for word in its source'))
                    out.steps.append(s)
            elif not out.steps: why = 'a stroke before the first step'
            elif len(out.steps[-1]['strokes']) >= LIMITS['strokes']: why = 'more than 6 strokes in a step'
            else: out.steps[-1]['strokes'].append(_stroke(o, ctx))
        if why == META_WHY: out.dropped.append((k, why)); skip = True; continue
        skip = False if isinstance(o, dict) and o.get('op') == 'step' else skip
        if why:
            out.dropped.append((k, why)); run += 1
            if run >= 2: out.failed = True; break
        else: run = 0
    _pair(out, ctx)
    _shape(out)
    _counts(out.steps)
    for st in out.steps:   # a row with `at` draws its own circle: a circle line on the same thing goes
        on = [json.dumps(s['at'], sort_keys=True) for s in st['strokes'] if s['op'] == 'row' and 'at' in s]
        if (two := [s for s in st['strokes'] if s['op'] == 'circle' and json.dumps(s['at'], sort_keys=True) in on]):
            st['strokes'] = [s for s in st['strokes'] if s not in two]; out.dropped += [(-1, 'the row draws its own circle')] * len(two)
    return out

The default check of a quote: white space does not count, a part of a word does not count, and a short quote does not count:

In [ ]:
text = 'A notable property of the forward process is that it admits sampling x_t at an arbitrary timestep t in closed form.'
test_eq(verbatim('it admits  sampling x_t at an arbitrary timestep t in closed form', text), 'it admits sampling x_t at an arbitrary timestep t in closed form')
test_eq(verbatim('t admits sampling x_t at an arbitrary timestep t in closed form', text), '')
test_eq(verbatim('closed form', text), '')

A board for "I don't get alpha bar" on the DDPM formula: three steps, each with its strokes. The header says that the
tutor is sure.

In [ ]:
reply = '\n'.join(json.dumps(o) for o in [
    dict(op='v', n=1, sure=0.9, need='none'),
    dict(op='step', say='This formula adds noise in one jump.'), dict(op='box', at='m1', w=2),
    dict(op='step', say='Alpha bar t is how much of the clean picture is left at step t.'),
    dict(op='circle', at=dict(a='m1', tex=r'\bar\alpha_t'), w=0), dict(op='label', at=dict(a='m1', tex=r'\bar\alpha_t'), text='how much is left', side='above', w=5),
    dict(op='step', say='It starts near one and falls to near zero.'), dict(op='sketch', at=dict(a='figure', pt=[1050, 0]), pts=[[0, 100], [500, 500], [1000, 950]])])
b = parse_board(reply, ctx)
test_eq(len(b.steps), 3)
test_eq([len(s['strokes']) for s in b.steps], [1, 2, 1])
test_eq((b.sure, b.need, b.failed, b.dropped), (0.9, 'none', False, []))

A bad line drops alone, and the next line still draws. Two bad lines in a row end the board:

In [ ]:
b = parse_board('{"op":"step","say":"One."}\n{"op":"circle","at":"e7"}\n{"op":"circle","at":"m1"}', ctx)
test_eq((len(b.steps[0]['strokes']), len(b.dropped), b.failed), (1, 1, False))
b = parse_board('{"op":"step","say":"One."}\nnot json\n{"op":"circle","at":"e7"}\n{"op":"step","say":"Two."}', ctx)
test_eq((len(b.steps), b.failed), (1, True))

A step that talks about the board, not the idea, drops with its strokes, such as an excuse for an earlier offer. It does not end the board:

In [ ]:
meta = '\n'.join(json.dumps(o) for o in [dict(op='step', say='You asked to see it move, but this page has no live demo I can start; my earlier offer was misleading.'),
                                         dict(op='box', at='passage'), dict(op='step', say='Alpha bar t is how much is left.'), dict(op='circle', at='m1')])
b = parse_board(meta, ctx)
test_eq(([s['say'] for s in b.steps], b.steps[0]['strokes'], b.failed), (['Alpha bar t is how much is left.'], [dict(op='circle', at='m1')], False))
test_eq([w for _, w in b.dropped], [META_WHY, 'a stroke of a dropped step'])
test_eq([bool(META.search(s)) for s in ('Sorry, I cannot show that.', 'I apologise for the mix-up.', 'The noise is not misleading here.', 'The last step adds noise.')], [True, True, False, False])

A row with `at` draws its own circle, so a circle line on the same thing drops. A row comes to the page with its cells as text. A semicolon in a spoken line becomes the end of a sentence:

In [ ]:
both = '\n'.join(json.dumps(o) for o in [dict(op='step', say='Alpha bar sets the mix; 18 goes in and 47 comes out.'), dict(op='circle', at=dict(a='passage', text='closed form')),
                                         dict(op='row', cells=[18], to=47, at=dict(a='passage', text='closed form'))])
b = parse_board(both, ctx)
test_eq((b.steps[0]['say'], b.steps[0]['strokes'], b.dropped), ('Alpha bar sets the mix. 18 goes in and 47 comes out.', [dict(op='row', cells=['18'], to='47', at=dict(a='passage', text='closed form'))],
                                                               [(-1, 'the row draws its own circle')]))

A row of a worked example with the answer in a last cell, and no `to`, gets its answer after the arrow. A chain stays a chain, and in its panel a pair with an answer becomes a chain too (see "One shape for each panel" below). A head that names "four stages" over fewer rows loses the count:

In [ ]:
b2 = parse_board('\n'.join(json.dumps(o) for o in [dict(op='step', say='Work it out from 1.0: guess, compare and update the weights.'), dict(op='panel', head='Repeat these four stages', numbered=True),
    dict(op='row', cells=['1.0 and 1.0', '√0.5 × 1 + √0.5 × 1', '1.414']), dict(op='panel', head='Then learn'), dict(op='row', cells=['Guess', 'Compare'], chain=True), dict(op='row', cells=['Update'], to='weights')]), ctx)
rs = [s for s in b2.steps[0]['strokes'] if s['op'] == 'row']
test_eq([(r['cells'], r.get('to'), r.get('chain', False)) for r in rs], [(['1.0 and 1.0', '√0.5 × 1 + √0.5 × 1'], '1.414', False), (['Guess', 'Compare'], None, True), (['Update', 'weights'], None, True)])
test_eq(b2.steps[0]['strokes'][0]['head'], 'Repeat these stages')

The key terms of a line: words cut to 5 letters, numbers and maths tokens. A small word and the article "a" do not count:

In [ ]:
test_eq(key_terms('Multiply each $x_i$ by $w_i$ to get w_1x_1 and 47'), {'multi', 'x_i', 'w_i', 'w_1x_1', '47'})
test_eq(key_terms(r'Add the products: \\sum of a list'), {'add', 'produ', 'sum', 'list'})

A row is named when its step shares a word (in any form) or a symbol with it. The single letter x does not name `x_i`:

In [ ]:
test_eq(named(dict(op='row', cells=['Add products, then b'], to='z'), 'Then add the products and the bias b.'), True)
test_eq(named(dict(op='row', cells=['Multiply each x_i by w_i'], to='w_1x_1, …, w_nx_n'), 'Numbers x go in, one number z comes out.'), False)

The reply of the screenshot that a tester sent, on the idea "Functions and the neuron": step 1 says that numbers go in and one
number comes out, but its row multiplies each input by its weight. After the check, that row moves to step 2, whose line names
it. Step 1 keeps its circle, because a stream sends that circle before the row comes:

In [ ]:
nctx = by_id(dict(anchors=[dict(id='passage', kind='passage', text='A neuron takes numbers x_1, ..., x_n, multiplies each by a weight w_i, adds a bias b, and outputs z = w_1x_1 + ... + w_nx_n + b.',
                                tokens=['x_1', 'w_i', 'b', 'z', 'w_1x_1'])]))
w1 = dict(a='passage', tex='w_1x_1')
neuron = [dict(op='v', n=1, sure=0.9, need='none'),
    dict(op='step', say='This page asks you to see a neuron as a function: numbers x go in, one number z comes out.'),
    dict(op='panel', title='Inside one neuron', head='Turn many inputs into one output', cols=['Step', 'Result'], numbered=True),
    dict(op='circle', at=w1), dict(op='row', cells=['Multiply each x_i by w_i'], to='w_1x_1, …, w_nx_n', at=w1, w=4),
    dict(op='step', say='First it multiplies each input by its weight, then adds the products and the bias b.'),
    dict(op='row', cells=['Add products, then b'], to='z'),
    dict(op='step', say='The output z then goes on to the next layer.'),
    dict(op='row', cells=['Layer 1 outputs z'], to='Layer 2 inputs x')]
b = parse_board('\n'.join(json.dumps(o) for o in neuron), nctx)
test_eq([[s['op'] for s in st['strokes']] for st in b.steps], [['panel', 'circle'], ['row', 'row'], ['row']])
test_eq(b.steps[1]['strokes'][0], dict(op='row', cells=['Multiply each x_i by w_i'], to='w_1x_1, …, w_nx_n', at=w1))

Here "z" names the row of step 3, so it stays. With a line that names nothing of it, the row drops:

In [ ]:
neuron[-2] = dict(op='step', say='That is the whole neuron.')
b = parse_board('\n'.join(json.dumps(o) for o in neuron), nctx)
test_eq(([[s['op'] for s in st['strokes']] for st in b.steps], (-1, 'a row that its step does not name') in b.dropped), ([['panel', 'circle'], ['row', 'row'], []], True))

A well-formed reply, where each line names its row, stays as it is:

In [ ]:
good_n = [neuron[0], dict(op='step', say='A neuron is a function: numbers x go in, and one number z comes out.'),
    dict(op='panel', title='Inside one neuron', head='Turn many inputs into one output', numbered=True),
    dict(op='step', say='First it multiplies each input x i by its weight w i.'), dict(op='row', cells=['Multiply each $x_i$ by $w_i$'], to='$w_1x_1, …, w_nx_n$', at=w1),
    dict(op='step', say='Then it adds the products and the bias b, and that sum is z.'), dict(op='row', cells=['Add the products and $b$'], to='$z$')]
b = parse_board('\n'.join(json.dumps(o) for o in good_n), nctx)
test_eq((b.dropped, [[s['op'] for s in st['strokes']] for st in b.steps]), ([], [['panel'], ['row'], ['row']]))

A row that names nothing on the page drops, also when its line names it: a second layer and its softmax are later ideas on a page about one neuron.

In [ ]:
later = [dict(op='step', say='Layer 2 then runs a softmax.'), dict(op='panel', head='Then'), dict(op='row', cells=['Layer 2'], to='softmax')]
b = parse_board('\n'.join(json.dumps(o) for o in later), nctx)
test_eq(([s['op'] for s in b.steps[0]['strokes']], b.dropped), (['panel'], [(-1, 'a row that names nothing on the page')]))

A row does not move past the next panel line, because that line starts a new example:

In [ ]:
two = [dict(op='step', say='Numbers go in.'), dict(op='panel', head='One'), dict(op='row', cells=['Multiply'], to='w_1x_1'),
       dict(op='step', say='A new example.'), dict(op='panel', head='Two'), dict(op='step', say='Multiply them.')]
test_eq([[s['op'] for s in st['strokes']] for st in parse_board('\n'.join(json.dumps(o) for o in two), ctx).steps], [['panel'], ['panel'], []])

The board of a screenshot from the owner, on the step "Auto-regressive encoder-decoder transduction" of "Attention Is All You
Need". Its panel has the column names "From" and "To", and its rows have three shapes: pairs, chains of three words, and a
note. The names stood over no cells, and nothing lined up. A small context with the passage and its formulas:

In [ ]:
actx = by_id(dict(anchors=[dict(id='passage', kind='passage', text='Here, the encoder maps an input sequence of symbol representations $(x_1, \dots, x_n)$ to a sequence of continuous representations $\mathbf{z} = (z_1, \dots, z_n)$. Given $\mathbf{z}$, the decoder then generates an output sequence $(y_1, \dots, y_m)$ of symbols one element at a time.'),
                           dict(id='m1', kind='formula', text=r'(x_1, \dots, x_n)', tokens=['x_1', 'x_n'])]))
att = [dict(op='step', say='The encoder turns the input symbols x_1 to x_n into a list z.'),
       dict(op='panel', title='Turn input into output', head='Symbols to symbols', cols=['From', 'To']),
       dict(op='row', cells=['Input symbols'], to='Output symbols'), dict(op='row', cells=[r'$x_1,\dots,x_n$', 'Encoder', r'$z=(z_1,\dots,z_n)$'], chain=True),
       dict(op='step', say='The decoder reads z and gives the output symbols y_1 to y_m.'), dict(op='row', cells=['$z$', 'Decoder', r'$y_1,\dots,y_m$'], chain=True),
       dict(op='step', say='Earlier outputs go back into the decoder, so they help shape the next output.'),
       dict(op='row', cells=['Earlier outputs'], to='Decoder'), dict(op='note', text='Earlier outputs help shape the next output.')]
rows = lambda b: [(r['cells'], r.get('to'), r.get('chain', False)) for st in b.steps for r in st['strokes'] if r['op'] == 'row']

The first row is a pair that fits the two names, so the names stay, and the two chains of three words drop. Each row that stays has one cell and its answer:

In [ ]:
b = parse_board('\n'.join(json.dumps(o) for o in att), actx)
test_eq(rows(b), [(['Input symbols'], 'Output symbols', False), (['Earlier outputs'], 'Decoder', False)])
test_eq(b.steps[0]['strokes'][0]['cols'], ['From', 'To'])
test_eq(b.dropped, [(-1, 'a row of another shape than the first row of its panel')] * 2)

When the chain comes first, the panel is a flow: the names go, and each pair with an answer becomes a chain of two words. No row drops:

In [ ]:
flow = [att[0], att[1], att[3], att[2], *att[4:]]
b = parse_board('\n'.join(json.dumps(o) for o in flow), actx)
test_eq([len(c) for c, to, chain in rows(b)], [3, 2, 3, 2])
assert all(chain and to is None for c, to, chain in rows(b))
test_eq(('cols' in b.steps[0]['strokes'][0], b.dropped), (False, [(-1, 'column names that do not fit the first row')]))

A chain of two words fits a pair panel as one cell and its answer. A pair with two cells does not fit a panel of pairs with one cell:

In [ ]:
pp = [dict(op='step', say='Guess, then compare the guess.'), dict(op='panel', head='Guess and compare', cols=['Guess', 'Compare']),
      dict(op='row', cells=['Guess'], to='compare'), dict(op='row', cells=['Guess', 'Compare'], chain=True), dict(op='row', cells=['Guess', 'Compare'], to='update')]
b = parse_board('\n'.join(json.dumps(o) for o in pp), ctx)
test_eq(rows(b), [(['Guess'], 'compare', False), (['Guess'], 'Compare', False)])
test_eq(b.dropped, [(-1, 'a row of another shape than the first row of its panel')])

A panel shows at most 6 rows. The seventh drops:

In [ ]:
seven = [dict(op='step', say='Guess and compare.'), dict(op='panel', head='Guess'), *[dict(op='row', cells=['Guess'], to=f'compare {k}') for k in range(4)],
         dict(op='step', say='Guess again and compare.'), *[dict(op='row', cells=['Guess'], to=f'compare {k}') for k in range(4, 7)]]
b = parse_board('\n'.join(json.dumps(o) for o in seven), ctx)
test_eq((len(rows(b)), b.dropped), (6, [(-1, 'more than 6 rows in a panel')]))

Some models put the strokes inside the step. They count as the lines after it:

In [ ]:
b = parse_board('{"op":"step","say":"One.","strokes":[{"op":"circle","at":"m1"},{"op":"box","at":"m1"}]}', ctx)
test_eq(([s['op'] for s in b.steps[0]['strokes']], b.dropped), (['circle', 'box'], []))

At most 6 strokes in a step and 6 steps in a board; a stroke before the first step drops:

In [ ]:
many = '\n'.join(['{"op":"step","say":"One."}'] + ['{"op":"circle","at":"m1"}'] * 6 + ['{"op":"box","at":"m1"}', '{"op":"step","say":"Two."}'])
b = parse_board(many, ctx)
test_eq(([len(s['strokes']) for s in b.steps], len(b.dropped)), ([6, 0], 1))
test_eq(len(parse_board('\n'.join(['{"op":"step","say":"A step."}'] * 7), ctx).steps), 6)
test_eq(parse_board('{"op":"circle","at":"m1"}\n{"op":"step","say":"One."}', ctx).dropped[0][1], 'a stroke before the first step')

The count of steps follows the reply: a board of 1, 2 or 5 steps keeps that count. Only a seventh step drops:

In [ ]:
def steps(n): return '\n'.join(json.dumps(dict(op='step', say=f'Part {k} of the idea.')) for k in range(1, n + 1))
test_eq([len(parse_board(steps(n), ctx).steps) for n in (1, 2, 5, 7)], [1, 2, 5, 6])

A quote stays only when it is word for word in its source:

In [ ]:
qctx = by_id({**ctx, 'sources': [dict(title='DDPM', text='A notable property of the forward process is that it admits sampling x_t at an arbitrary timestep t in closed form.')]})
good = dict(op='step', say='The paper says it.', src=1, quote='it admits sampling x_t at an arbitrary timestep t in closed form')
test_eq(parse_board(json.dumps(good), qctx).steps[0]['quote']['src'], 1)
b = parse_board(json.dumps({**good, 'quote': 'it allows sampling x_t at any timestep t in closed form, fast'}), qctx)
assert 'quote' not in b.steps[0] and b.dropped

A host can give its own check of a quote. This one takes no quote:

In [ ]:
test_eq('quote' in parse_board(json.dumps(good), qctx, verbatim=lambda q, t: '').steps[0], False)

A `video_at` stroke moves to the start of its cue and gets the link to that time:

In [ ]:
b = parse_board('\n'.join([json.dumps(dict(op='step', say='The video says it.')), json.dumps(dict(op='video_at', t=66, quote='share of the SIGNAL', w=2))]), vctx)
test_eq(b.steps[0]['strokes'], [dict(op='video_at', t=61.5, quote='share of the signal', link='https://www.youtube.com/watch?v=abc123def45&t=61s', w=2)])

A line of more than 600 bytes drops. There is no diagram: a picture of an idea is a side panel, so a diagram line is an unknown op:

In [ ]:
test_eq(parse_board(json.dumps(dict(op='step', say='Long.')) + '\n' + json.dumps(dict(op='label', at='passage', text='x' * 40, side='left' + ' ' * 700)), ctx).dropped[0][1],
        'not one JSON object of 600 bytes or less')
test_eq([w for _, w in parse_board(json.dumps(dict(op='step', say='A tree.')) + '\n' + json.dumps(dict(op='diagram', nodes=[], edges=[])), ctx).dropped], ['an unknown op'])

### The context

The page sends what it shows: its address, the step, the concept, the anchors, the demo, the video, and what the
learner showed with the pointer (the focus), and the thread: the last questions on this page.
`clean_context` keeps only the keys that it knows and cuts the text: at most 40 anchors, 300 characters an anchor and
6000 in all. An anchor id has only letters, digits and `:_-`. An unknown kind becomes `text`. A `try` anchor is the last graded try
of the open check (see *The learner's answer*): it keeps its own fields, each with its own bound (`TRY`). The result is a `PageContext`. A host cleans each context that is not one (abhyasa's `explain` does), so a raw dict is
cleaned too, even when it has a `by` key.

In [ ]:
#| export
KINDS = ('passage', 'formula', 'figure', 'caption', 'note', 'check', 'try', 'demo', 'input', 'readout', 'video', 'title', 'code', 'text')
ANCHOR_ID = re.compile(r'\A[A-Za-z0-9:_-]{1,40}\Z')
MOST = dict(anchors=40, text=300, total=6000, tokens=40, token=120, labels=20)
TRY = dict(answer=400, feedback=500, error=240, outcome=40, gaps=5, gap=60, left=9)   # the bounds of the last graded try of a check

def _s(x, n:int) -> str: return ' '.join(str(x).split())[:n] if isinstance(x, (str, int, float)) and not isinstance(x, bool) else ''

def _labels(d) -> dict:
    "The good label boxes of `d`: a short name to a box from 0 to 1000."
    if not isinstance(d, dict): return {}
    return {k[:12]: v for k, v in list(d.items())[:MOST['labels']] if isinstance(k, str) and k.strip() and _box(v)}

def _try(a) -> dict:
    """The fields of a `try` anchor: the learner's `answer`, the `outcome`, the `feedback`, the `error` line and the `gaps` that the
    try points at, each cut to size; the `state` of the check (`open` or `closed`), and the tries `left`."""
    out = {k: v for k in ('answer', 'outcome', 'feedback', 'error') if (v := _s(a.get(k, ''), TRY[k]))}
    gs = [g for g in (_s(x, TRY['gap']) for x in (a.get('gaps') if isinstance(a.get('gaps'), list) else [])[:20]) if g]
    if gs: out['gaps'] = list(dict.fromkeys(gs))[:TRY['gaps']]
    out['state'] = 'closed' if a.get('state') == 'closed' else 'open'
    if isinstance(a.get('left'), int) and not isinstance(a['left'], bool) and 0 <= a['left'] <= TRY['left']: out['left'] = a['left']
    return out

def _anchor(a) -> dict|None:
    if not isinstance(a, dict) or not isinstance(a.get('id'), str) or not ANCHOR_ID.match(a['id']): return None
    out = dict(id=a['id'], kind=a.get('kind') if a.get('kind') in KINDS else 'text', text=_s(a.get('text', ''), MOST['text']))
    if out['kind'] == 'try':
        out.update(_try(a))
        if not (out.get('answer') or out.get('feedback')): return None   # a try with no answer and no feedback says nothing
    if isinstance(a.get('tokens'), list):
        out['tokens'] = list(dict.fromkeys(t[:MOST['token']] for t in a['tokens'] if isinstance(t, str) and t.strip()))[:MOST['tokens']]
    if (ls := _labels(a.get('labels'))): out['labels'] = ls
    return out

#### What the learner shows

While the learner talks or types the question, they can circle a part with the mouse, or hold the pointer still on it. The host
finds the gestures and sends them as `focus`: a list of items, the latest first. Each item has a `kind` (`circle` or `point`), the
`anchors` under it (the best one first), a `box` from 0 to 1000 of its first anchor, and `t`, the milliseconds from the start of
the question. An item can have `words`: what the learner said or typed near that time.

`_focus` keeps at most 5 good items. It keeps only the anchor ids that the page sent, and drops an item with none of them, with a
bad kind or with a bad box.

In [ ]:
#| export
FOCUS = dict(items=5, anchors=4, words=80)
SHOWN = ('circle', 'point')

def _focus(raw, ids) -> list:
    "The good items of `raw` that the learner showed with the pointer, with only the anchor ids in `ids`. At most 5, in the order of the host."
    out = []
    for f in raw if isinstance(raw, list) else ():
        if len(out) >= FOCUS['items']: break
        if not isinstance(f, dict) or f.get('kind') not in SHOWN or not _box(f.get('box')) or not isinstance(f.get('anchors'), list): continue
        box, got = [round(x) for x in f['box']], list(dict.fromkeys(a for a in f['anchors'] if isinstance(a, str) and a in ids))[:FOCUS['anchors']]
        if not got or not _box(box): continue
        o = dict(kind=f['kind'], box=box, anchors=got)
        if _num(f.get('t')) and f['t'] >= 0: o['t'] = round(f['t'])
        if (w := _s(f.get('words', ''), FOCUS['words'])): o['words'] = w
        out.append(o)
    return out

#### The thread of a page

A learner who asks "say that more simply" or "why?" after a board means that board. So the page keeps a short thread in memory,
and sends it with the next question: the last 3 turns on this page, each with the question (`said`), the lines that its board said
(`says`) and its tier (`t`). The page forgets the thread when the page changes, after 10 minutes with no question, or when the learner
pushes Start fresh. The server keeps no thread: each question brings its own.

`_thread` keeps the good turns, the oldest first: a question of at most 300 characters, at most 6 lines of at most 160 characters,
and a known tier. A turn with no question or no line drops, and so does a turn with a tag or a link in a line, which a board never
says. At most 1500 characters stay in all: the oldest turn goes first.

In [ ]:
#| export
THREAD = dict(turns=3, said=300, says=6, say=160, total=1500)
TIERS = ('tutor', 'vault', 'web')

def _turn(t) -> dict|None:
    "One good turn of a thread: the question `said`, the lines `says` of its board, and its tier `t`. None when it is bad."
    if not isinstance(t, dict) or not (said := _s(t.get('said', ''), THREAD['said'])) or not isinstance(t.get('says'), list): return None
    says = [s for s in (_s(x, THREAD['say']) for x in t['says'][:THREAD['says']]) if s]
    if not says or any(UNSAFE.search(s) for s in says): return None
    return dict(said=said, says=says, t=t.get('t') if t.get('t') in TIERS else 'tutor')

def _size(t) -> int: return len(t['said']) + sum(map(len, t['says']))

def _thread(raw) -> list:
    "The good turns of `raw`, the oldest first: the last 3, with at most 1500 characters in all. The oldest turn goes first."
    out = [t for t in map(_turn, raw[-10:] if isinstance(raw, list) else ()) if t][-THREAD['turns']:]
    while out and sum(map(_size, out)) > THREAD['total']: out.pop(0)
    return out

The demo part keeps the inputs (with their value, range and label), the readouts, and whether the gate is open.

In [ ]:
#| export
def _demo(d) -> dict:
    if not isinstance(d, dict): return {}
    ins = {}
    for k, v in list((d.get('inputs') or {}).items())[:20]:
        if not (isinstance(k, str) and ANCHOR_ID.match(k) and isinstance(v, dict)): continue
        ins[k] = {f: v[f] for f in ('min', 'max', 'step') if _num(v.get(f))}
        if isinstance(v.get('value'), (int, float, str, bool)): ins[k]['value'] = v['value'] if not isinstance(v['value'], str) else v['value'][:60]
        if v.get('label'): ins[k]['label'] = _s(v['label'], 60)
    return dict(slug=_s(d.get('slug', ''), 80), inputs=ins, open=d.get('open') is True,
                readouts=[_s(r, 120) for r in (d.get('readouts') or [])[:8] if _s(r, 120)])

class PageContext(AttrDict):
    "A page context that `clean_context` made. A host cleans every other context, also one with a `by` key."

def clean_context(raw) -> PageContext:
    "The page context `raw`, with only known keys, cut to size, and with `by` (its anchors by id)."
    raw = raw if isinstance(raw, dict) else {}
    anchors, total = [], 0
    for a in (raw.get('anchors') or [])[:MOST['anchors']]:
        if (a := _anchor(a)) is None: continue
        a['text'] = a['text'][:max(0, MOST['total'] - total)]
        total += len(a['text'])
        anchors.append(a)
    ctx = dict(page=_s(raw.get('page', ''), 200), title=_s(raw.get('title', ''), 200), step=_s(raw.get('step', ''), 200),
               concept=_s(raw.get('concept', ''), 200), slug=_s(raw.get('slug', ''), 80), source=_s(raw.get('source', ''), 300),
               anchors=anchors, demo=_demo(raw.get('demo')), leela=raw.get('leela') is True)
    if (fs := _focus(raw.get('focus'), {a['id'] for a in anchors})): ctx['focus'] = fs
    if (th := _thread(raw.get('thread'))): ctx['thread'] = th
    if isinstance(raw.get('video'), dict) and isinstance(raw['video'].get('id'), str): ctx['video'] = dict(id=_s(raw['video']['id'], 11))
    return PageContext(by_id(ctx))

Unknown keys go, and a bad anchor id goes:

In [ ]:
c = clean_context(dict(page='/walk/x/step/1', onload='x', anchors=[dict(id='passage', kind='passage', text='Some  text.'), dict(id='<b>', text='x')]))
test_eq(([a['id'] for a in c.anchors], c.anchors[0]['text'], 'onload' in c), (['passage'], 'Some text.', False))

A raw dict with a `by` key is not a `PageContext`, so a host cleans it. A cleaned context stays the same type:

In [ ]:
test_eq((isinstance(c, PageContext), isinstance(dict(c), PageContext)), (True, False))

A circle on the formula and a point on the passage stay, the latest first. An unknown anchor id goes from its item:

In [ ]:
shown = [dict(kind='circle', box=[100, 50, 400, 900], anchors=['m1', 'e7', 'passage'], t=2100.4, words='why is this  here'),
         dict(kind='point', box=[480, 480, 520, 520], anchors=['passage'], t=800)]
c = clean_context(dict(anchors=ctx.anchors, focus=shown))
test_eq(c.focus, [dict(kind='circle', box=[100, 50, 400, 900], anchors=['m1', 'passage'], t=2100, words='why is this here'),
                  dict(kind='point', box=[480, 480, 520, 520], anchors=['passage'], t=800)])

Bad items drop: an unknown kind, no known anchor, a box that is not four finite numbers from 0 to 1000, and a box that is not a list:

In [ ]:
bad = [dict(kind='lasso', box=[0, 0, 10, 10], anchors=['m1']), dict(kind='point', box=[0, 0, 10, 10], anchors=['e7']),
       dict(kind='point', box=[0, 0, float('nan'), 10], anchors=['m1']), dict(kind='point', box=[0, 0, 2000, 10], anchors=['m1']),
       dict(kind='point', box='0,0,10,10', anchors=['m1']), dict(kind='point', box=[5, 0, 5, 10], anchors=['m1']),
       dict(kind='point', box=[0, 0, 10, 10], anchors='m1'), 'circle']
test_eq('focus' in clean_context(dict(anchors=ctx.anchors, focus=bad)), False)
test_eq('focus' in clean_context(dict(anchors=ctx.anchors, focus='m1')), False)

At most 5 items stay, and each keeps at most 4 anchors:

In [ ]:
many = [dict(kind='point', box=[0, 0, 10, 10], anchors=['m1', 'passage', 'figure', 'm1'], t=k) for k in range(8)]
c = clean_context(dict(anchors=ctx.anchors, focus=many))
test_eq(([f['t'] for f in c.focus], c.focus[0]['anchors']), ([0, 1, 2, 3, 4], ['m1', 'passage', 'figure']))

A good thread stays, the oldest first. A long line is cut to 160 characters, and an unknown tier is the tutor:

In [ ]:
th = [dict(said='I do not get alpha bar', says=['Alpha bar is the share that is left.', 'x' * 200], t='tutor'),
      dict(said='why?', says=['Each step keeps a part of the signal.'], t='magic')]
c = clean_context(dict(anchors=ctx.anchors, thread=th))
test_eq([(t['said'], len(t['says'][1]) if len(t['says']) > 1 else 0, t['t']) for t in c.thread], [('I do not get alpha bar', 160, 'tutor'), ('why?', 0, 'tutor')])

A bad thread drops: a thread that is not a list, a turn with no question or no line, and a line with a tag or a link:

In [ ]:
for bad in ('say it again', dict(said='x'), [dict(said='', says=['a'])], [dict(said='x', says=[])], [dict(said='x', says='a line')],
            [dict(said='x', says=['<script>alert(1)</script>'])], [dict(said='x', says=['see https://example.org'])], [7, None]):
    assert 'thread' not in clean_context(dict(anchors=ctx.anchors, thread=bad)), bad

At most the last 3 turns stay, and at most 1500 characters in all. The oldest turn goes first:

In [ ]:
many = [dict(said=f'question {k}', says=['a line']) for k in range(5)]
test_eq([t['said'] for t in clean_context(dict(thread=many)).thread], ['question 2', 'question 3', 'question 4'])
big = [dict(said=f'q{k}', says=['y' * 160] * 6) for k in range(3)]
th = clean_context(dict(thread=big)).thread
test_eq(([t['said'] for t in th], sum(map(_size, th)) <= THREAD['total']), (['q2'], True))

#### The learner's answer

The page can send the last graded try of the check as the anchor `try`: the learner's answer, the
outcome, the feedback, the Error line, the gaps that the answer points at, the state of the check and the tries that are left.
Each field is cut to its bound, an unknown state is `open`, and a try with no answer and no feedback drops:

In [ ]:
TRY_RAW = dict(id='try', kind='try', text='Alpha bar grows with t.', answer='Alpha bar grows with t, because more noise is added.',
               outcome='Not yet', feedback='Look at what each alpha is: a number below one.', error='Alpha bar does not grow: it is a product of numbers below one.',
               gaps=['cumulative product', 'cumulative product', 'noise schedule'], state='open', left=2)
a = clean_context(dict(anchors=[TRY_RAW])).anchors[0]
test_eq((a['kind'], a['gaps'], a['state'], a['left']), ('try', ['cumulative product', 'noise schedule'], 'open', 2))
a = clean_context(dict(anchors=[dict(TRY_RAW, answer='x' * 900, state='odd', left=True, gaps='not a list')])).anchors[0]
test_eq((len(a['answer']), a['state'], 'left' in a, 'gaps' in a), (TRY['answer'], 'open', False, False))
test_eq(clean_context(dict(anchors=[dict(id='try', kind='try', text='x')])).anchors, [])

## The video cues

A page can show a video with its transcript. The transcript is a list of cues `[start, text]`: `video_cues` makes them from fossick's
cues `{start, end, text}`. A `video_at` line quotes the transcript at its time (see *The check of one line*). `video_id` reads the
id of a YouTube link or id, as fossick reads it. It refuses a link to another site and an id that is not 11 characters.

In [ ]:
#| export
_VID = re.compile(r'[A-Za-z0-9_-]{11}')
_YT = re.compile(r'https?://(?:[a-z0-9-]+\.)?(?:youtube\.com|youtu\.be|youtube-nocookie\.com)/', re.I)

def video_id(url_or_id:str) -> str:
    "The id of a YouTube link or id, as fossick's `yt_link` reads it. Raise `ValueError` when it names no video."
    s = str(url_or_id or '').strip()
    if not s or len(s) > 300 or (not _VID.fullmatch(s) and not _YT.match(s)): raise ValueError('This is not a YouTube link.')
    m = re.search(r'(?:v=|youtu\.be/|/shorts/|/embed/)([A-Za-z0-9_-]{11})(?![A-Za-z0-9_-])', s)
    v = m[1] if m else s
    if not _VID.fullmatch(v): raise ValueError('This is not a YouTube link.')
    return v

def video_cues(cues) -> list:
    "fossick's cues `{start, end, text}` as the board's `[start, text]`, with empty lines dropped."
    return [[round(float(c['start']), 2), ' '.join(str(c['text']).split())] for c in cues or ()
            if isinstance(c, dict) and _num(c.get('start')) and str(c.get('text') or '').strip()]

A link, a short link and a bare id give the same id. Another site does not:

In [ ]:
for u in ('https://www.youtube.com/watch?v=abc123def45&t=3s', 'https://youtu.be/abc123def45', 'abc123def45'): test_eq(video_id(u), 'abc123def45')
test_fail(lambda: video_id('https://example.com/watch?v=abc123def45'))
test_fail(lambda: video_id('https://www.youtube.com/watch?v=short'))
test_eq(video_cues([dict(start=1.234, end=3, text=' Hello  there '), dict(start=4, end=5, text='')]), [[1.23, 'Hello there']])
test_eq(video_id('https://www.youtube.com/shorts/abc123def45'), 'abc123def45')
test_eq(yt_link('abc123def45', 61.5), 'https://www.youtube.com/watch?v=abc123def45&t=61s')

### The board as it comes

A board takes some seconds to write. So the host streams it, and the first step draws while the model writes the next one.
`stream_board` reads the reply as it comes. When a new line is in, it reads the whole lines so far again with `parse_board`, so
the rules are the same as for a whole reply. When the header says `sure` of `SURE` or more, `on_step(step, k)` gets each step
as soon as its spoken line and its first mark are in, and again each time it gets more strokes: so the first mark does not wait for
the rest of the step (the rows of a side panel) or for the next step. The page adds the strokes that it does not have yet. A step that is not sure, or a reply with no header,
sends nothing: the host can ask again with sources (abhyasa looks in the vault, then on the web). The board that `stream_board` returns is the truth:
its first steps are the steps that went to `on_step`.

When the stream breaks after a step went out, the board keeps the whole lines that came, so the page does not lose what it shows.

The learner can stop: a newer question, Esc, or a closed tab. Then the host sets the `threading.Event` `stop`. `stream_board`
reads no more chunks and closes the stream. So the model call ends at once, and the next question does not wait for the backend.

In [ ]:
#| export
def _held(steps) -> list:
    "`steps` with no panel line that has column names and no row yet, and none of its notes: its first row decides whether the names stay."
    ss = [s for st in steps for s in st['strokes'] if s['op'] in PANEL]
    wait = set()
    for k, s in enumerate(ss):
        if s['op'] != 'panel' or 'cols' not in s: continue
        beat = list(takewhile(lambda x: x['op'] != 'panel', ss[k + 1:]))
        if not any(x['op'] == 'row' for x in beat): wait |= {id(s), *map(id, beat)}
    return [{**st, 'strokes': [s for s in st['strokes'] if id(s) not in wait]} for st in steps] if wait else steps

def stream_board(chunks, ctx, on_step=None, sure:float=SURE, stop=None, verbatim=verbatim) -> AttrDict:
    """`parse_board` of the reply that comes in `chunks`. When the header says `sure` or more, `on_step(step, k)` gets each step as soon as
    its first mark is in, and again each time its strokes change, so the first mark does not wait for the rows of a panel or for the next
    step. A panel line with column names waits for its first row (see `_shape`). The page adds the strokes that it does not have yet. When the `threading.Event` `stop` is set, it reads no more chunks and closes
    the stream, so the model call ends and lets go of its backend. `verbatim` checks a quote, as in `parse_board`."""
    text, sent, seen = '', {}, -1   # the strokes of each step that went out, as JSON
    def send(b, whole):
        if not on_step or (b.sure or 0) < sure: return
        for k, s in enumerate(b.steps if whole else _held(b.steps), 1):
            if k == len(b.steps) and not whole and not s['strokes']: break   # the newest step goes out with its first mark
            if sent.get(k) != (j := json.dumps(s['strokes'], sort_keys=True)): sent[k] = j; on_step(s, k)
    it = iter(chunks)
    try:
        for c in it:
            if stop is not None and stop.is_set(): break
            text += c
            if on_step and (n := text.rfind('\n')) > seen: seen = n; send(parse_board(text[:n], ctx, verbatim), False)
    except Exception:
        if not sent: raise
        text = text[:text.rfind('\n') + 1]   # the reply broke after a step went out: keep its whole lines
    finally:
        if hasattr(it, 'close'): it.close()
    b = parse_board(text, ctx, verbatim)
    if stop is None or not stop.is_set(): send(b, True)
    return b

A sure board of three steps, for the tests:

In [ ]:
three = '\n'.join(json.dumps(o) for o in [dict(op='v', n=1, sure=0.9), dict(op='step', say='Alpha bar t is the share that is left.'),
                                          dict(op='circle', at=dict(a='m1', tex=r'\bar\alpha_t')), dict(op='step', say='Its root scales x0.'),
                                          dict(op='box', at='m1'), dict(op='step', say='The rest is noise.'), dict(op='underline', at='passage')])

Each step goes out before the last chunk comes, and the last copy of each step is the step of the board:

In [ ]:
events, got = [], {}
def chunks(t, n=7):
    for i in range(0, len(t), n): events.append('chunk'); yield t[i:i + n]
b = stream_board(chunks(three), ctx, lambda s, k: (got.__setitem__(k, s), events.append(f'step {k}')))
assert events.index('step 1') < len(events) - 1 - events[::-1].index('chunk'), events
test_eq(([got[k] for k in (1, 2, 3)], b.steps[0]['strokes']), (b.steps, [dict(op='circle', at=dict(a='m1', tex=r'\bar\alpha_t'))]))

A board that is not sure sends no step. Its board is the same as the board of the whole reply:

In [ ]:
got = []
low = three.replace('0.9', '0.4')
test_eq((stream_board(chunks(low), ctx, lambda s, k: got.append(k)).steps, got), (parse_board(low, ctx).steps, []))

A broken line in the middle drops as `parse_board` drops it, and the next step still goes out:

In [ ]:
board_rows = three.split('\n')
broken = '\n'.join(board_rows[:3] + ['{"op":"circle","at":"e7"}'] + board_rows[3:])
got = []
last = {}
b = stream_board((broken[i:i + 5] for i in range(0, len(broken), 5)), ctx, lambda s, k: last.__setitem__(k, s))
whole = parse_board(broken, ctx)
test_eq(([last[k] for k in sorted(last)], b.steps, b.dropped), (whole.steps, whole.steps, whole.dropped))
test_eq((len(last), len(b.dropped)), (3, 1))

When the stream breaks in step 3, after step 1 went out, the board keeps the whole lines that came: two steps. A break before any step raises, as a call does:

In [ ]:
def breaks(text, at):
    yield text[:at]
    raise RuntimeError('the model stopped')
got = []
b = stream_board(breaks(three, three.index('The rest') + 3), ctx, lambda s, k: got.append(k))
test_eq((sorted(set(got)), len(b.steps)), ([1, 2], 2))
test_fail(lambda: stream_board(breaks(three, 5), ctx, lambda s, k: None), contains='the model stopped')

A step with a side panel goes out with its first stroke, before its rows and before the next step. Then it goes out again as its rows come:

In [ ]:
pan = '\n'.join(json.dumps(o) for o in [dict(op='v', n=1, sure=0.9), dict(op='step', say='Each test is a question: 1.0 gives 1.414, and 2.0 gives 2.0.'), dict(op='panel', head='Make a question'),
    dict(op='row', cells=['[1.0]'], to='1.414', at='passage'), dict(op='row', cells=['[2.0]'], to='2.0'), dict(op='step', say='Then it adds up.')]) + '\n'
seen_ = []
stream_board((pan[i:i + 4] for i in range(0, len(pan), 4)), ctx, lambda s, k: seen_.append((k, len(s['strokes']))))
test_eq(seen_, [(1, 1), (1, 2), (1, 3), (2, 0)])

A panel line with column names waits for its first row, because that row decides whether the names stay. The step still goes out with its first mark:

In [ ]:
held = '\n'.join(json.dumps(o) for o in att[:3]) + '\n'
seen_ = []
stream_board((held[i:i + 4] for i in range(0, len(held), 4)), actx, lambda s, k: seen_.append([x['op'] for x in s['strokes']]), sure=0)
test_eq(seen_, [['panel', 'row']])
seen_ = []
b = stream_board((held[i:i + 4] for i in range(0, len(held) - 60, 4)), actx, lambda s, k: seen_.append([x['op'] for x in s['strokes']]), sure=0)
test_eq((seen_, [s['op'] for s in b.steps[0]['strokes']]), ([['panel']], ['panel']))   # a reply that ends before the row: the panel goes out at the end

The stream keeps the pairing: the row of the screenshot does not go out with step 1. It goes out with step 2, whose line
names it, and step 1 never gets it:

In [ ]:
sent = []
neuron_r = '\n'.join(json.dumps(o) for o in neuron[:7]) + '\n'
stream_board((neuron_r[i:i + 9] for i in range(0, len(neuron_r), 9)), nctx, lambda s, k: sent.append((k, [x['op'] for x in s['strokes']])))
test_eq(any(k == 1 and 'row' in ops for k, ops in sent), False)
test_eq(sent[-1], (2, ['row', 'row']))

A set `stop` ends the read at the next chunk, and the stream closes. No more step goes out:

In [ ]:
import threading

In [ ]:
stop, got, shut = threading.Event(), [], []
def slow():
    try:
        for i in range(0, len(three), 7):
            if i > three.index('Its root') + 30: stop.set()   # the learner stops while step 2 comes
            yield three[i:i + 7]
    finally: shut.append(True)
b = stream_board(slow(), ctx, lambda s, k: got.append(k), stop=stop)
test_eq((got, shut), ([1], [True]))

A host gives its own check of a quote to the stream too, and the stream gives it to each read:

In [ ]:
said = []
stream_board(chunks(json.dumps(good)), qctx, verbatim=lambda q, t: said.append(q) or '')
assert said